# Prompt Engineering Tools

**Module:** 07-prompt-engineering

**Notebook:** `10-prompt-engineering-tools.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Tooling Landscape** with clear contracts and failure modes
- Explain and apply **LangSmith** with clear contracts and failure modes
- Explain and apply **PromptLayer** with clear contracts and failure modes
- Explain and apply **OpenAI Playground** with clear contracts and failure modes
- Explain and apply **Anthropic Console** with clear contracts and failure modes
- Explain and apply **Google AI Studio** with clear contracts and failure modes
- Explain and apply **Humanloop** with clear contracts and failure modes
- Explain and apply **Practical Workflow** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Prompt Engineering Tools

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Tooling Landscape**
2. **LangSmith**
3. **PromptLayer**
4. **OpenAI Playground**
5. **Anthropic Console**
6. **Google AI Studio**
7. **Humanloop**
8. **Practical Workflow**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Tooling Landscape

### Definition
**Tooling Landscape** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Tooling Landscape typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tooling Landscape: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Tooling Landscape as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tooling Landscape as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tooling Landscape
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Tooling Landscape when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Tooling Landscape improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Tooling Landscape" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tooling Landscape"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## LangSmith

### Definition
**LangSmith** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around LangSmith typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For LangSmith: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain LangSmith as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating LangSmith as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for LangSmith
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use LangSmith when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "LangSmith" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "LangSmith"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "LangSmith"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "LangSmith"}
strong = {"definition": "LangSmith", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "LangSmith"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "LangSmith", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — LangSmith

**Situation:** A team wants to productionize a feature involving **LangSmith**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## PromptLayer

### Definition
**PromptLayer** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around PromptLayer typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For PromptLayer: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain PromptLayer as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating PromptLayer as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for PromptLayer
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use PromptLayer when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "PromptLayer" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "PromptLayer"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "PromptLayer"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "PromptLayer"}
strong = {"definition": "PromptLayer", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "PromptLayer"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "PromptLayer", "passed": len(checks)-len(failed), "failed": failed})


## OpenAI Playground

### Definition
**OpenAI Playground** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around OpenAI Playground typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For OpenAI Playground: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain OpenAI Playground as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating OpenAI Playground as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for OpenAI Playground
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use OpenAI Playground when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "OpenAI Playground" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "OpenAI Playground"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


### Worked scenario — OpenAI Playground

**Situation:** A team wants to productionize a feature involving **OpenAI Playground**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Anthropic Console

### Definition
**Anthropic Console** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Anthropic Console typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Anthropic Console: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Anthropic Console as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Anthropic Console as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Anthropic Console
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Anthropic Console when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Anthropic Console" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Anthropic Console"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


## Google AI Studio

### Definition
**Google AI Studio** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Google AI Studio typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Google AI Studio: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Google AI Studio as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Google AI Studio as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Google AI Studio
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Google AI Studio when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Google AI Studio" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Google AI Studio"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Google AI Studio"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Google AI Studio"}
strong = {"definition": "Google AI Studio", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Google AI Studio"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Google AI Studio", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Google AI Studio

**Situation:** A team wants to productionize a feature involving **Google AI Studio**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Humanloop

### Definition
**Humanloop** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Humanloop typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Humanloop: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Humanloop as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Humanloop as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Humanloop
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Humanloop when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Humanloop" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Humanloop"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Humanloop"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Humanloop"}
strong = {"definition": "Humanloop", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Humanloop"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Humanloop", "passed": len(checks)-len(failed), "failed": failed})


## Practical Workflow

### Definition
**Practical Workflow** is a core building block in 10-prompt-engineering-tools within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Practical Workflow typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Practical Workflow: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Practical Workflow as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Practical Workflow as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Practical Workflow
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Practical Workflow when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Practical Workflow" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Practical Workflow"
    notebook: str = "10-prompt-engineering-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_7 = ConceptContract()
print(json.dumps({"contract": asdict(contract_7), "health": contract_7.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Practical Workflow"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Practical Workflow"}
strong = {"definition": "Practical Workflow", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Practical Workflow"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Practical Workflow", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Practical Workflow

**Situation:** A team wants to productionize a feature involving **Practical Workflow**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Prompt Engineering Tools**.

| Topic | Do | Don't |
|-------|----|-------|
| Tooling Landscape | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| LangSmith | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| PromptLayer | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| OpenAI Playground | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Anthropic Console | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Google AI Studio | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Tooling Landscape | Key concept covered in this notebook; see its section for definition and pitfalls |
| LangSmith | Key concept covered in this notebook; see its section for definition and pitfalls |
| PromptLayer | Key concept covered in this notebook; see its section for definition and pitfalls |
| OpenAI Playground | Key concept covered in this notebook; see its section for definition and pitfalls |
| Anthropic Console | Key concept covered in this notebook; see its section for definition and pitfalls |
| Google AI Studio | Key concept covered in this notebook; see its section for definition and pitfalls |
| Humanloop | Key concept covered in this notebook; see its section for definition and pitfalls |
| Practical Workflow | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Prompt Engineering Tools** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **07-prompt-engineering**.


## Try It Yourself

1. Implement a failing test/fixture for **Tooling Landscape**, then fix your demo until it passes.
2. Implement a failing test/fixture for **LangSmith**, then fix your demo until it passes.
3. Implement a failing test/fixture for **PromptLayer**, then fix your demo until it passes.
4. Implement a failing test/fixture for **OpenAI Playground**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Anthropic Console**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
